![](https://github.com/datagong/data/blob/main/datagong.png?raw=true)

© DATAGONG - Tous droits réservés - 2026

📋 **Rappels Conditions Générales d'Utilisation**

⚠️ Les Notebooks sont privés

❌ partage des Notebooks, de leur contenu, des liens, des images, …

❌ publier les Notebooks sur GitHub (public) ou tout autre outil de versionning

✅ sauvegarder les Notebooks dans votre environnement personnel et privé

✅ réutiliser les codes dans le cadre de vos projets en entreprise / projets personnels / etc.

✅ annoter / modifier les Notebooks dans votre environnement personnel et privé

# <center><u><b>Data Visualization avec Streamlit et Plotly</b></u></center>

# <b>Streamlit + Plotly — 6. Dashboard : KPI, tables & cross‑filtres</b>

Dans le notebook précédent, vous avez structuré votre application en multi‑pages avec colonnes, onglets et navigation automatique. Votre projet a désormais l'allure d'un vrai outil professionnel.

Il est temps de passer à l'étape suivante : construire une **page Dashboard** dédiée, avec des indicateurs chiffrés (KPI), un graphique de tendance, un tableau de données filtrable et un bouton d'export. C'est exactement le type de vue que vos futurs utilisateurs métier s'attendent à trouver dans une application de data visualisation.

# 0. Créer un dashboard de métriques avec Streamlit

Un dashboard, c'est avant tout une page qui répond en un coup d'œil à la question : **« où en est-on ? »**. Pour y arriver, nous allons assembler plusieurs briques Streamlit que nous découvrirons une par une : un titre, un filtre, des métriques, un graphique, un tableau et un bouton d'export.

Chaque section ci‑dessous vous présente la méthode Streamlit associée, puis nous assemblerons le tout dans le code final.

## 0.1. Ajoutez un titre et des sous-titres

Tout bon dashboard commence par un titre clair. Streamlit fournit deux méthodes pratiques pour hiérarchiser vos contenus :

[`st.title`](https://docs.streamlit.io/develop/api-reference/text/st.title) affiche un titre principal en gros caractères, idéal pour nommer votre page. [`st.subheader`](https://docs.streamlit.io/develop/api-reference/text/st.subheader) ajoute un sous-titre plus discret, parfait pour séparer les différentes sections de votre dashboard (graphique, tableau, etc.). Ces deux méthodes prennent simplement une chaîne de caractères en paramètre.

```python
st.title("📊 Dashboard")
st.subheader("Tendance (catégorie)")
```

## 0.2. Sélection de filtres interactifs

Pour qu'un dashboard soit utile, il faut que l'utilisateur puisse choisir **ce qu'il veut voir**. Nous allons donc ajouter un filtre grâce à [`st.selectbox`](https://docs.streamlit.io/develop/api-reference/widgets/st.selectbox), que vous avez déjà croisée dans les notebooks précédents.

Pour rappel, `st.selectbox` affiche une liste déroulante et renvoie la valeur sélectionnée. Ses principaux paramètres sont `label` (le texte affiché au-dessus du sélecteur), `options` (la liste des choix possibles) et `index` (l'option sélectionnée par défaut).

```python
cat = st.selectbox("Catégorie", options=sorted(list(df["categorie"].cat.categories)))
```

Ici, nous récupérons les catégories directement depuis le DataFrame et nous les trions par ordre alphabétique. La valeur choisie par l'utilisateur est stockée dans la variable `cat`, que nous utiliserons ensuite pour filtrer les données.

## 0.3. Affichage des indicateurs clés (KPI)

Les KPI — *Key Performance Indicators* — sont les chiffres que l'on veut voir apparaître en haut du dashboard : chiffre d'affaires total, moyenne des ventes, valeur maximale… Bref, les métriques qui résument la situation en un coup d'œil.

Streamlit propose pour cela [`st.metric`](https://docs.streamlit.io/develop/api-reference/data/st.metric). Cette méthode affiche un grand nombre bien lisible, accompagné d'un libellé. Vous pouvez aussi lui passer un paramètre `delta` pour montrer une variation (par exemple +12 % par rapport au mois précédent) — Streamlit colorera automatiquement la valeur en vert ou rouge selon le signe.

```python
st.metric("Moyenne ventes", f"{df_cat['ventes'].mean():.1f}")
st.metric("Max ventes", f"{df_cat['ventes'].max():.0f}")
```

💡 Pour disposer plusieurs KPI côte à côte, pensez à utiliser `st.columns` que vous connaissez déjà — nous le ferons dans le code final.

## 0.4. Visualisation des données

Vous avez déjà créé des graphiques Plotly dans les notebooks précédents. Pour les intégrer dans Streamlit, nous utilisons [`st.plotly_chart`](https://docs.streamlit.io/develop/api-reference/charts/st.plotly_chart). Cette méthode prend en paramètre un objet figure Plotly (celui que vous construisez avec `px.line`, `px.bar`, ou nos fonctions utilitaires `make_line` / `make_bar`) et l'affiche directement dans la page. Le paramètre `use_container_width=True` est très pratique : il force le graphique à occuper toute la largeur disponible, ce qui évite les problèmes de mise en page.

```python
st.plotly_chart(make_line(df_cat, "date", "ventes", title=f"Tendance — {cat}"), use_container_width=True)
```

## 0.5. Tableaux interactifs

Un graphique donne la tendance, mais parfois vos utilisateurs veulent aussi explorer les données ligne par ligne. C'est exactement le rôle de [`st.dataframe`](https://docs.streamlit.io/develop/api-reference/data/st.dataframe) : cette méthode affiche un DataFrame Pandas sous forme de tableau interactif, avec tri par colonne et défilement intégrés — sans rien coder de plus. Comme pour le graphique, `use_container_width=True` ajuste la largeur du tableau à celle du conteneur.

```python
st.dataframe(df_cat, use_container_width=True)
```

## 0.6. Export et téléchargement de données

Dernier ingrédient, et pas des moindres : permettre à l'utilisateur de **récupérer les données qu'il vient de filtrer**. Dans un contexte métier, c'est souvent indispensable — un analyste voudra exporter un sous-ensemble de données pour le retravailler dans Excel ou le joindre à un e‑mail.

Streamlit propose pour cela [`st.download_button`](https://docs.streamlit.io/develop/api-reference/widgets/st.download_button). Cette méthode crée un bouton cliquable qui déclenche le téléchargement d'un fichier. Vous devez lui fournir le contenu à télécharger via le paramètre `data` (ici, un CSV encodé en UTF‑8), un `file_name` pour nommer le fichier, et un type `mime` pour que le navigateur sache comment le traiter.

```python
csv = df_cat.to_csv(index=False).encode("utf-8")
st.download_button("Télécharger CSV filtré", data=csv, file_name=f"ventes_{cat}.csv", mime="text/csv")
```

Toutes les briques sont en place. Exécutez la cellule ci-dessous pour générer la page `Dashboard.py`, puis relancez votre application Streamlit pour voir le résultat.

In [1]:
%%writefile ../streamlit_app/pages/Dashboard.py
import streamlit as st
from utils.data import load_data, filter_data
from utils.charts import make_line

# Titre du dashboard
st.title("📊 Dashboard")

# Chargement des données
df = load_data()

# Sélecteur de catégorie (filtre interactif)
cat = st.selectbox("Catégorie", options=sorted(list(df["categorie"].cat.categories)))

# Filtrage des données selon la catégorie sélectionnée
df_cat = filter_data(df, categorie=[cat])

# Affichage des indicateurs clés (KPI)
col1, col2 = st.columns(2)
with col1:
    st.metric("Moyenne ventes", f"{df_cat['ventes'].mean():.1f}")
with col2:
    st.metric("Max ventes", f"{df_cat['ventes'].max():.0f}")

# Affichage du graphique de tendance
st.subheader("Tendance (catégorie)")
st.plotly_chart(make_line(df_cat, "date", "ventes", title=f"Tendance — {cat}"), use_container_width=True)

# Affichage du tableau filtré
st.subheader("Table filtrée")
st.dataframe(df_cat, use_container_width=True)

# Bouton de téléchargement du CSV filtré
csv = df_cat.to_csv(index=False).encode("utf-8")
st.download_button("Télécharger CSV filtré", data=csv, file_name=f"ventes_{cat}.csv", mime="text/csv")

Writing ../streamlit_app/pages/Dashboard.py


# <font color='#ff7373'><b>Félicitations !</b></font>

Vous venez de construire un vrai dashboard fonctionnel — KPI, graphique, tableau et export — en quelques lignes de code. Votre application commence sérieusement à ressembler à un outil d'aide à la décision.

📖 Il existe bien d'autres composants pour enrichir vos dashboards. N'hésitez pas à explorer la [documentation officielle de Streamlit](https://docs.streamlit.io/develop/api-reference) pour découvrir les widgets que nous n'avons pas abordés ici.

Dans le prochain notebook, nous ajouterons des **fonctionnalités d'export et de partage** pour aller encore plus loin.

# <center><font color='#3b4859'><u>![](https://github.com/datagong/data/blob/main/mini%20datagong%202.png?raw=true)</u></font></center>